In [1]:
import pandas as pd

file_path = r"C:\Users\JasonChen\OneDrive - Vena\Product Ops & Analytics\Advanced Analytics\flat file sources\integrations_relationships_0401v1.csv"
df = pd.read_csv(file_path)

df.columns = df.columns.str.upper()
df['ACCOUNT_TYP_NM'] = df['ACCOUNT_TYP_NM'].str.upper()

drop_types = {"PARTNER", "PARTNER PROSPECT", "PROSPECT"}
df = df[~df["ACCOUNT_TYP_NM"].isin(drop_types)]

# Existing pipeline...
pivot = pd.pivot_table(
    df,
    values="ACCOUNT_ID",
    index="INTEGRATION_STATUS",
    columns="ACCOUNT_TYP_NM",
    aggfunc=pd.Series.nunique,
    fill_value=0
)

pivot = pivot.copy()
pivot['CUSTOMER_TO_FORMER_CUSTOMER_RATIO'] = pivot.get('CUSTOMER', 0) / pivot.get('FORMER CUSTOMER', 1)

# New block: churn probability summary stats by integration status
churn_summary = df.groupby('INTEGRATION_STATUS')['CHURN_PROBABILITY'].agg(
    average='mean',
    median='median',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75)
).reset_index()

pivot
churn_summary
##render 

,INTEGRATION_STATUS,average,median,p25,p75
0,established integration method,0.183051,0.07,0.02,0.24
1,flat file or unidentified method,0.260199,0.11,0.03,0.43


In [3]:

import pandas as pd

file_path = r"C:\Users\JasonChen\OneDrive - Vena\Product Ops & Analytics\Advanced Analytics\flat file sources\integrations_relationships_0401v1.csv"
df = pd.read_csv(file_path)

df.columns = df.columns.str.upper()
df['ACCOUNT_TYP_NM'] = df['ACCOUNT_TYP_NM'].str.upper()

drop_types = {"PARTNER", "PARTNER PROSPECT", "PROSPECT"}
df = df[~df["ACCOUNT_TYP_NM"].isin(drop_types)]

# Existing pipeline...
pivot = pd.pivot_table(
    df,
    values="ACCOUNT_ID",
    index="INTEGRATION_STATUS",
    columns="ACCOUNT_TYP_NM",
    aggfunc=pd.Series.nunique,
    fill_value=0
)

pivot = pivot.copy()
pivot['CUSTOMER_TO_FORMER_CUSTOMER_RATIO'] = pivot.get('CUSTOMER', 0) / pivot.get('FORMER CUSTOMER', 1)

pivot



ACCOUNT_TYP_NM,CUSTOMER,FORMER CUSTOMER,CUSTOMER_TO_FORMER_CUSTOMER_RATIO
INTEGRATION_STATUS,,,
established integration method,1586,429,3.696970
flat file or unidentified method,533,503,1.059642


In [10]:
# %%
import pandas as pd

file_path = r"C:\Users\JasonChen\OneDrive - Vena\Product Ops & Analytics\Advanced Analytics\flat file sources\integrations_relationships_0401v1.csv"

df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.upper()

bool_cols = [
    "HAS_CMD_LINE",
    "HAS_POWER_AUTOMATE",
    "HAS_IMPORT_API",
    "HAS_NATIVE_CONNECTOR"
]

# Normalize boolean values safely
for col in bool_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(["TRUE", "1", "YES", "Y"])
    )

# Create result dict
result = {}

for col in bool_cols:
    temp = (
        df[df[col]]  # only rows where flag = True
        .groupby("ACCOUNT_TYP_NM")["ACCOUNT_ID"]
        .nunique()
    )
    result[col] = temp

# Combine into one dataframe
pivot_df = pd.DataFrame(result).fillna(0).astype(int)

pivot_df

,HAS_CMD_LINE,HAS_POWER_AUTOMATE,HAS_IMPORT_API,HAS_NATIVE_CONNECTOR
ACCOUNT_TYP_NM,,,,
Customer,670,239,428,775
Former Customer,200,61,34,206
Partner,0,0,1,0
Partner Prospect,1,0,1,1
